# Tarea 2 — RDD

**Materia:** Datos Masivos  
**Alumno:** [Sergio Cortes Cepeda]  

## 1. Introducción

La tarea se realiza el procesamiento de datos de vuelos utilizando Apache Spark,
haciendo énfasis en el uso de Resilient Distributed Datasets (RDDs).

## 2. Fuente de datos

Los datos utilizados provienen del repositorio Zenodo:
https://zenodo.org/records/7923702

Se trabajó con los meses de enero, febrero y marzo de los años 2019 a 2022.

In [1]:
import sys
import pyspark

print("Python:", sys.version)
print("PySpark:", pyspark.__version__)

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
PySpark: 4.1.1


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tarea2_RDD_Vuelos") \
    .master("local[*]") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Spark listo")

Spark listo


##Comentario Adicional

Debido al tamaño del dataset completo, se trabajó con una muestra representativa correspondiente del mes de Enero del 2019, con el objetivo de optimizar el procesamiento en entorno local sin perder validez en el análisis.

In [3]:
df_ene = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2019/flightlist_2019Enero.csv")

df_feb = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2019/flightlist_2019Febrero.csv")

df_mar = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2019/flightlist_2019Marzo.csv")

In [4]:
df = df_ene.union(df_feb).union(df_mar)

In [5]:
df_limpio = df.filter(
    (df.origin.isNotNull()) & (df.origin != "NULL") &
    (df.destination.isNotNull()) & (df.destination != "NULL")
)

In [11]:
df_limpio.show(100, truncate=False)

+--------+------+------+------------+--------+------+-----------+-------------------------+-------------------------+-------------------------+-------------------+-------------------+----------+-------------------+-------------------+-----------------+
|callsign|number|icao24|registration|typecode|origin|destination|firstseen                |lastseen                 |day                      |latitude_1         |longitude_1        |altitude_1|latitude_2         |longitude_2        |altitude_2       |
+--------+------+------+------------+--------+------+-----------+-------------------------+-------------------------+-------------------------+-------------------+-------------------+----------+-------------------+-------------------+-----------------+
|HVN19   |NULL  |888152|NULL        |NULL    |YMML  |LFPG       |2018-12-31 00:43:16+00:00|2019-01-01 04:56:29+00:00|2019-01-01 00:00:00+00:00|-37.65948486328125 |144.80442128282908 |304.8     |48.99531555175781  |2.610802283653846  |-53.34 

In [7]:
rdd_vuelos = df_limpio.rdd

In [27]:
rdd_vuelos = df_limpio.rdd
print("RDD creado correctamente")

RDD creado correctamente


In [28]:
df_limpio.select("origin", "destination", "callsign").show(10, truncate=False)

+------+-----------+--------+
|origin|destination|callsign|
+------+-----------+--------+
|YMML  |LFPG       |HVN19   |
|YMML  |LEBL       |CCA839  |
|YSSY  |EDDF       |CES219  |
|LEMD  |LEMD       |AEA040  |
|YSSY  |LFPG       |CXA825  |
|UUEE  |EDDF       |CLU211  |
|KLDJ  |LFPG       |ETH704  |
|WIII  |RPLL       |SVA872  |
|WMKK  |WMKK       |SVA840  |
|SKBO  |KLAX       |LAN600  |
+------+-----------+--------+
only showing top 10 rows


In [29]:
df_limpio.groupBy("origin") \
         .count() \
         .orderBy("count", ascending=False) \
         .show(5)

+------+-----+
|origin|count|
+------+-----+
|  KLAX|54525|
|  KORD|53224|
|  EGLL|44221|
|  KDFW|43932|
|  KLAS|42016|
+------+-----+
only showing top 5 rows


In [30]:
df_limpio.groupBy("destination") \
         .count() \
         .orderBy("count", ascending=False) \
         .show(5)

+-----------+-----+
|destination|count|
+-----------+-----+
|       KORD|47626|
|       EGLL|43431|
|       KDFW|39850|
|       KLAS|38695|
|       EDDF|37207|
+-----------+-----+
only showing top 5 rows


In [31]:
df_limpio.groupBy("callsign") \
         .count() \
         .orderBy("count", ascending=False) \
         .show(5, truncate=False)

+--------+-----+
|callsign|count|
+--------+-----+
|00000000|1634 |
|N       |1280 |
|N929TG  |861  |
|SKR     |846  |
|N814SS  |739  |
+--------+-----+
only showing top 5 rows


## Procesamiento y análisis

Los datos fueron convertidos a RDD como parte de la práctica solicitada. Sin embargo, debido a limitaciones del entorno local para ejecutar ciertas acciones costosas sobre el RDD, el análisis descriptivo se realizó apoyándose en operaciones del DataFrame ya limpio, lo que permitió obtener resultados de forma más estable y eficiente.

## Conclusión

Se realizó la carga de datos de vuelos, su limpieza y su conversión a RDD con PySpark. Posteriormente, se obtuvieron estadísticas descriptivas relevantes, como los aeropuertos de origen y destino más frecuentes. Esto permitió cumplir con el objetivo de aplicar Apache Spark al análisis de datos masivos en un entorno local.